# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

**Objective.** Tim produk e-commerce ingin membangun fitur otomatis yang mengklasifikasikan sentimen ulasan pelanggan pada halaman produk. Eksperimen ini membandingkan dua kandidat pendekatan (model klasik Scikit-learn vs LLM API Gemini) pada data yang sama, agar keputusan pendekatan untuk produksi didasarkan pada data, bukan asumsi.

**Target/label.** Kolom `sentiment` dengan dua kelas: `positif` dan `negatif` (klasifikasi biner). Input yang digunakan hanya kolom `review_text` (teks ulasan Bahasa Indonesia, gaya informal).

**Batasan & asumsi.**
- Hanya menggunakan dataset yang dibagikan (`customer_reviews_sentiment.csv`, 200 baris). Dataset tidak diubah dan tidak ditambah.
- Tidak ada kelas netral; setiap ulasan dianggap jelas positif atau negatif.
- Kedua pendekatan dievaluasi pada test set yang sama (split 80/20, stratified, `random_state=42`).
- LLM digunakan apa adanya tanpa fine-tuning, hanya melalui prompt; model diakses lewat free tier Gemini API sehingga terikat rate limit.
- Deep learning dari nol berada di luar cakupan.

## 2. Import Library & Load Dataset

In [1]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
from dotenv import load_dotenv
import os
import time

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


In [2]:
print('Shape:', df.shape)
print('Missing values:', df.isna().sum().sum())
print('Jumlah teks ulasan unik:', df['review_text'].nunique())
df['sentiment'].value_counts()

Shape: (200, 4)
Missing values: 0
Jumlah teks ulasan unik: 40


sentiment
positif    110
negatif     90
Name: count, dtype: int64

## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [3]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 160
Test size: 40


In [4]:
# Cek seberapa banyak kalimat test yang juga muncul persis di train (dataset banyak berisi kalimat berulang)
overlap = X_test.isin(set(X_train)).mean()
print(f'Proporsi teks test yang juga ada di train: {overlap:.0%}')

Proporsi teks test yang juga ada di train: 98%


**Catatan dataset.** Dataset hanya memiliki 40 kalimat ulasan unik yang diulang pada produk berbeda, dan 98% kalimat di test set juga muncul persis di train set. Artinya model klasik pada dasarnya "sudah pernah melihat" hampir semua soal ujian, sehingga skor model klasik lebih mencerminkan kemampuan menghafal daripada generalisasi. Sebaliknya, LLM tidak pernah melihat train set sama sekali (zero-shot), sehingga skornya murni kemampuan bawaan model. Hal ini penting untuk membaca hasil di Section 6.

## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [5]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [6]:
start = time.perf_counter()
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)
train_time_classic = time.perf_counter() - start
print(f'Waktu training: {train_time_classic:.3f} detik')

Waktu training: 0.031 detik


### 4.3 Prediksi pada Test Set

In [7]:
start = time.perf_counter()
y_pred_classic = clf.predict(X_test_vec)
infer_time_classic = time.perf_counter() - start
print(f'Waktu inferensi {len(X_test)} ulasan: {infer_time_classic:.4f} detik')

Waktu inferensi 40 ulasan: 0.0014 detik


## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [8]:
load_dotenv() 
assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY belum di-set (lihat README)'

client = genai.Client(
    api_key=os.environ.get('GEMINI_API_KEY'),
    # retry otomatis jika terkena rate limit (429) atau error sementara dari server
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(attempts=5, initial_delay=5)),
)

### 5.2 Merancang Prompt

Prompt yang digunakan (**zero-shot**):

```
Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:
```

**Alasan desain prompt:**
- **Zero-shot** dipilih karena task-nya sederhana (biner) dan ulasan umumnya eksplisit, sehingga LLM tidak memerlukan contoh. Zero-shot juga membuat prompt lebih pendek (token lebih sedikit, biaya lebih rendah) dan menjadi perbandingan yang adil sebagai pendekatan "tanpa training / tanpa data berlabel". Few-shot baru perlu dipertimbangkan jika muncul pola salah prediksi tertentu (misal sarkasme).
- Label ditulis eksplisit dan output dibatasi **satu kata** agar mudah di-parse dan dicocokkan dengan label asli.
- Prompt ditulis dalam Bahasa Indonesia agar konsisten dengan bahasa ulasan dan label.
- Ulasan dibungkus tanda kutip agar batas antara instruksi dan data jelas.

**Parameter yang digunakan:**
- `model='gemini-3.5-flash-lite'`: model ringan dan cepat, cukup untuk klasifikasi biner. Model `gemini-2.0-flash` pada starter sudah tidak tersedia, dan `gemini-3.8-flash` hanya memberi kuota free tier 20 request/hari (kurang dari 40 data test).
- `temperature=0.0`: klasifikasi membutuhkan jawaban yang deterministik dan konsisten (input yang sama → label yang sama), bukan jawaban kreatif. Temperature rendah juga mengurangi kemungkinan output keluar dari format.


In [ ]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
def classify_sentiment_llm(review_text):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite', 
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text.strip().lower()

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [ ]:
start = time.perf_counter()
y_pred_llm = []
for review in X_test:
    pred = classify_sentiment_llm(review)
    y_pred_llm.append(pred)
infer_time_llm = time.perf_counter() - start

# Normalisasi output LLM agar konsisten dengan label 'positif'/'negatif'
def normalize_label(text):
    text = text.strip().strip('."\'').lower()
    if 'negatif' in text:
        return 'negatif'
    if 'positif' in text:
        return 'positif'
    return 'tidak_valid'

raw_llm = y_pred_llm
y_pred_llm = [normalize_label(p) for p in raw_llm]

print(f'Waktu inferensi {len(X_test)} ulasan: {infer_time_llm:.2f} detik')
print('Output mentah unik dari LLM:', sorted(set(raw_llm)))
print('Jumlah output tidak valid:', y_pred_llm.count('tidak_valid'))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Waktu inferensi 40 ulasan: 145.93 detik
Output mentah unik dari LLM: ['negatif', 'positif']
Jumlah output tidak valid: 0


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [11]:
print('Model Klasik (Scikit-learn)')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

Model Klasik (Scikit-learn)
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


**Interpretasi model klasik:**
- **Accuracy 1.00**: seluruh 40 ulasan test diprediksi benar.
- **Precision 1.00** (kedua kelas): setiap ulasan yang diprediksi positif memang positif, begitu juga negatif, sehingga tidak ada false alarm.
- **Recall 1.00** (kedua kelas): tidak ada ulasan positif/negatif yang terlewat.
- **F1-Score 1.00**: keseimbangan precision dan recall sempurna.
- **Confusion matrix** `[[18, 0], [0, 22]]`: 18 negatif dan 22 positif semuanya benar, tanpa false positive/negative.

Namun, angka sempurna ini perlu dibaca hati-hati: 98% kalimat test juga ada di train set, sehingga model hanya perlu "mengingat" kalimat yang sama. Skor ini **belum membuktikan** model mampu menangani ulasan baru dengan kosakata berbeda.

### 6.2 Evaluasi LLM API

In [12]:
print('LLM API (Gemini)')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

LLM API (Gemini)
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


**Interpretasi LLM API (Gemini):**
- **Accuracy 1.00**: seluruh 40 ulasan test diklasifikasikan benar.
- **Precision & Recall 1.00** (kedua kelas): tidak ada salah label maupun ulasan yang terlewat.
- **F1-Score 1.00**: performa seimbang di kedua kelas.
- **Confusion matrix** `[[18, 0], [0, 22]]`: identik dengan model klasik.
- Tidak ada output di luar format (0 output tidak valid), sehingga prompt satu kata + normalisasi bekerja dengan baik.

Berbeda dengan model klasik, hasil ini dicapai **tanpa training dan tanpa melihat train set**, sehingga lebih mencerminkan kemampuan generalisasi ke ulasan yang belum pernah dilihat.

### 6.3 Tabel Perbandingan Ringkasan

Precision, Recall, dan F1-Score menggunakan rata-rata **macro** (rata-rata kedua kelas tanpa pembobotan), karena kedua kelas sama pentingnya bagi use case ini.

In [13]:
from sklearn.metrics import precision_recall_fscore_support

def summarize(y_pred, infer_time):
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        y_test, y_pred, labels=['positif', 'negatif'], average='macro', zero_division=0
    )
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (macro)': precision,
        'Recall (macro)': recall,
        'F1-Score (macro)': f1_score,
        'Waktu inferensi test set (detik)': infer_time,
    }

comparison = pd.DataFrame({
    'Model Klasik (TF-IDF + LogReg)': summarize(y_pred_classic, infer_time_classic),
    'LLM API (Gemini, zero-shot)': summarize(y_pred_llm, infer_time_llm),
}).T
comparison.round(4)

,Accuracy,Precision (macro),Recall (macro),F1-Score (macro),Waktu inferensi test set (detik)
Model Klasik (TF-IDF + LogReg),1.0,1.0,1.0,1.0,0.0014
"LLM API (Gemini, zero-shot)",1.0,1.0,1.0,1.0,145.9300


Dari tabel di atas, kedua pendekatan sama-sama mendapat nilai 1.0 untuk accuracy, precision, recall, dan F1-score, jadi dari sisi metrik hasilnya seri.

Perbedaannya terlihat di waktu inferensi. Model klasik hanya butuh sekitar 0.0004 detik untuk memprediksi 40 ulasan, sedangkan Gemini butuh sekitar 145 detik karena rate limit free tier. Model klasik memang perlu training dulu dan butuh data berlabel, tapi proses training dengan dataset ini hanya sekitar 0.031 detik. 

Dari sisi biaya, model klasik berjalan di CPU lokal tanpa biaya tambahan. LLM dihitung per token untuk setiap ulasan yang diklasifikasikan, dan di free tier jumlah request-nya dibatasi kuota harian.

## 7. Analisis Trade-off dan Limitation

**Performa.** Kedua pendekatan mencapai Accuracy, Precision, Recall, dan F1-Score 1.00 pada test set yang sama, sehingga dari metrik saja tidak ada pemenang. Namun kualitas bukti keduanya berbeda: 98% kalimat test model klasik sudah muncul di train set (hasilnya dominan hafalan), sedangkan LLM bekerja zero-shot tanpa pernah melihat data. Jadi hasil LLM adalah indikator generalisasi yang lebih kuat.

**Effort implementasi.**
- Model klasik: butuh data berlabel, preprocessing (TF-IDF), training, dan retraining ketika pola ulasan berubah. Untuk dataset ini effort-nya kecil, tetapi di dunia nyata mengumpulkan dan melabeli ribuan ulasan yang beragam adalah pekerjaan terbesar.
- LLM API: tanpa training dan tanpa data berlabel; effort ada pada desain prompt, parsing/normalisasi output, penanganan error, rate limit, dan pengelolaan API key.

**Kecepatan.** Model klasik: training ±0.004 detik, inferensi 40 ulasan ±0.0004 detik secara lokal. LLM: ±124 detik untuk 40 ulasan (±3 detik/ulasan) karena latensi jaringan dan jeda rate limit free tier. Untuk klasifikasi real-time dalam jumlah besar, LLM jauh lebih lambat.

**Biaya.** Model klasik praktis gratis setelah dilatih (CPU biasa). LLM dikenakan biaya per token untuk setiap ulasan (±60 token input per request dengan prompt ini), sehingga biaya naik linear dengan jumlah ulasan. Free tier juga dibatasi kuota: saat eksperimen, `gemini-3.8-flash` hanya mengizinkan 20 request/hari, sehingga model harus diganti ke `gemini-3.5-flash-lite`.

Ulasan yang salah diprediksi oleh minimal satu pendekatan:

In [14]:
# Ulasan test yang salah diprediksi oleh minimal satu pendekatan
results = pd.DataFrame({'review_text': X_test, 'label': y_test, 'pred_klasik': y_pred_classic, 'pred_llm': y_pred_llm})
results[(results['label'] != results['pred_klasik']) | (results['label'] != results['pred_llm'])]

,review_text,label,pred_klasik,pred_llm


Tidak ada kesalahan prediksi yang teramati pada test set untuk kedua pendekatan (tabel di atas kosong). Meski begitu, beberapa limitation tetap terlihat dari eksperimen:

**Limitation model klasik:**
- Hanya mengenal kosakata dari 40 kalimat unik di train set. Ulasan baru dengan kata di luar vocabulary (slang, typo, singkatan seperti "brg", "mantul") tidak memiliki bobot, sehingga prediksinya tidak dapat diandalkan.
- TF-IDF bersifat bag-of-words dan tidak memahami urutan kata. Negasi seperti "tidak mengecewakan" (positif) berisiko terbaca negatif karena kata "mengecewakan", dan sarkasme sulit ditangkap. Pada data ini kalimat tersebut benar hanya karena kalimat yang sama sudah ada di train set.
- Skor 1.00 kemungkinan besar terlalu optimis (overfit/data leakage akibat duplikasi) dan tidak akan bertahan pada ulasan nyata yang lebih beragam.

**Limitation LLM API:**
- Bergantung pada layanan eksternal: latensi jaringan, rate limit, kuota harian, perubahan/penghentian model.
- Output berupa teks bebas sehingga tetap butuh normalisasi; walau pada eksperimen ini 0 output tidak valid, risikonya tetap ada.
- Biaya per request dan data ulasan dikirim ke pihak ketiga (pertimbangan privasi).
- Test set hanya 40 ulasan dengan kalimat yang eksplisit, sehingga kemampuan LLM pada kasus sulit (sarkasme, ulasan campuran positif-negatif) belum teruji di eksperimen ini.

## 8. Rekomendasi Technical Approach

**Rekomendasi: kembangkan pendekatan LLM API (Gemini, zero-shot) sebagai solusi awal produksi, sambil menyiapkan model klasik sebagai tahap optimasi biaya berikutnya.**

Alasan:
1. **Metrik seri, tetapi bukti LLM lebih kuat.** Keduanya 1.00 di semua metrik, namun 98% test set model klasik adalah kalimat yang sudah dihafal dari train set. LLM mencapai skor yang sama secara zero-shot, sehingga lebih bisa dipercaya untuk ulasan nyata yang jauh lebih beragam dari 40 kalimat template ini.
2. **Tidak butuh data berlabel.** Tim belum memiliki dataset ulasan berlabel yang representatif (hanya 40 kalimat unik). Membangun model klasik yang andal membutuhkan pengumpulan dan pelabelan data dalam jumlah besar terlebih dahulu, sedangkan LLM bisa langsung dipakai.
3. **Kecepatan bukan hambatan utama untuk use case ini.** Sentimen ulasan tidak perlu real-time; klasifikasi bisa dijalankan secara asynchronous/batch saat ulasan masuk, sehingga latensi ±1–3 detik per ulasan masih dapat diterima.

Catatan implementasi dan langkah lanjut:
- Gunakan akun berbayar (bukan free tier) agar tidak terkena kuota harian, dengan retry, timeout, dan normalisasi output seperti di notebook ini, serta fallback label/antrian ulang saat API gagal.
- Simpan hasil label LLM (dan koreksi manual jika ada) untuk membangun dataset berlabel yang besar dan beragam.
- Setelah data cukup, latih ulang model klasik dan evaluasi pada test set **tanpa duplikasi** dengan train set. Jika performanya mendekati LLM, pindah ke model klasik (atau pola hybrid: model klasik untuk ulasan dengan confidence tinggi, LLM hanya untuk kasus ragu) untuk menekan biaya dan latensi pada volume besar.